# Week 2 Notebook Lecture: First Data Workflow and Prompting for Python

This notebook is designed to be used in lieu of slides. It summarizes key ideas from the redesigned schedule and the original Python introduction lecture, then turns those ideas into a runnable data-cleaning workflow.

**Week 2 focus:** prompt design, generated-code review, simple Python/Pandas workflow, cleaned CSV output, and verification evidence.

**Instructor note:** The instructor can show prepared prompt cards and expected outputs without logging into a personal AI account. The code cells below can be run live, assigned for students to run, or used as reference examples.


## Runtime Dependency Note

These notebooks use standard Python plus `pandas`. Google Colab normally includes `pandas` already. If running in a local Jupyter environment, install or enable `pandas` before running the notebook. No Linux shell, server setup, paid cloud service, or private credential is required.


In [ ]:
# Run this early dependency check before the lecture demo cells.
try:
    import pandas as pd
except ImportError as exc:
    raise ImportError("This notebook needs pandas. Use Google Colab or install pandas in your local Jupyter environment.") from exc

print("pandas is available:", pd.__version__)


## Learning Objectives

By the end of Week 2, students should be able to:

- Explain why a useful AI prompt states task, input format, constraints, expected output, and verification checks.
- Read a small messy text/CSV-like sample into Python.
- Clean fields using beginner-friendly Python and pandas operations.
- Save a cleaned CSV and verify row count, column names, missing values, and sample records.
- Identify at least one weak assumption in generated or starter code.

**Success standard:** The cleaned file is not enough. Students must show evidence that the cleaned file is reasonable.


## Key Concept 1: Python Basics For Data Work

From the original Python introduction lecture, the Week 2 data workflow uses several fundamentals:

- **Variables:** names that store values, such as text, numbers, or DataFrames.
- **Strings:** text values used for names, categories, dates, and messy records.
- **Numbers:** integers and floats used for IDs, counts, prices, and measurements.
- **Comments:** notes that explain code for humans.
- **Conditionals:** logic for decisions such as missing vs. present.
- **Loops or vectorized operations:** repeated work across rows or fields.
- **Indentation:** Python uses indentation to show which lines belong together.

The notebook does not ask students to memorize all syntax at once. It asks students to run, inspect, and verify a small workflow.


In [ ]:
# Tiny Python basics check.

course_name = "Data Gathering and Warehousing"
week_number = 2
runtime_limit_minutes = 10

print(course_name)
print("Week:", week_number)
print("Required runtime limit:", runtime_limit_minutes, "minutes or less")
print("Type of course_name:", type(course_name))
print("Type of week_number:", type(week_number))


## Key Concept 2: A Prompt Is A Specification Draft

A weak prompt says:

> Clean this data.

A stronger prompt says:

> I am working in Google Colab free tier. Read this messy CSV-like text into pandas. The fields should be request_id, date, borough, complaint_type, status, and amount. Trim whitespace, standardize borough and status text, parse dates, convert amount to numeric, preserve missing values as NA, save a cleaned CSV, and print verification checks for row count, columns, missing values, and sample records. Avoid shell commands and keep the code beginner-friendly.

Why this is better:

- It names the environment.
- It gives the input and expected output structure.
- It names cleaning rules.
- It asks for verification.
- It states constraints.

AI can draft code, but students must still run it, read it, and revise it.


## Messy Mini Dataset

The sample below is deliberately small and imperfect. It imitates the kinds of issues that appear in public datasets:

- Extra spaces.
- Inconsistent capitalization.
- Missing values.
- Different date formats.
- Numeric values stored as text.
- Categories that need standardization.

This is demo-level on purpose. The goal is not scale; the goal is a complete evidence loop.


In [ ]:
from io import StringIO
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

messy_text = """request_id,date,borough,complaint_type,status,amount
1001,2026-01-03, Manhattan ,Noise, open ,12.50
1002,01/04/2026,BROOKLYN,Heat/Hot Water,CLOSED, 
1003,2026/01/05,Queens,Street Condition,Open,7
1004,not a date, bronx ,Noise,closed,4.25
1005,2026-01-06,,Water Leak,OPEN,11.00
1006,2026-01-07,Staten island,Noise,Pending,invalid
1007,2026-01-07,MANHATTAN,Noise,Open,15.75
"""

print(messy_text)


## Step 1: Read The Messy Data

Before cleaning, always inspect the raw shape and raw values.

The first read should answer:

- How many rows loaded?
- Which columns loaded?
- Did the parser understand the delimiter?
- Are obvious missing values present?


In [ ]:
raw_df = pd.read_csv(StringIO(messy_text))

print("Raw shape:", raw_df.shape)
print("Raw columns:", list(raw_df.columns))
raw_df


## Step 2: Inspect Problems Before Fixing Them

A common AI-generated-code weakness is jumping directly to cleaning without first showing the problem.

A better notebook makes the issues visible. This makes revisions easier and protects against accidental data changes.


In [ ]:
print("Data types before cleaning:")
print(raw_df.dtypes)

print("\nMissing values before cleaning:")
print(raw_df.isna().sum())

print("\nUnique borough values before cleaning:")
print(raw_df["borough"].dropna().unique())

print("\nUnique status values before cleaning:")
print(raw_df["status"].dropna().unique())


## Step 3: Clean Text Fields

For this demo, cleaning rules are intentionally simple:

- Strip leading/trailing spaces from text columns.
- Standardize borough names to title case.
- Standardize status to title case.
- Keep missing borough values missing.

A student should be able to explain every rule in plain language.


In [ ]:
clean_df = raw_df.copy()

text_columns = ["borough", "complaint_type", "status"]
for col in text_columns:
    clean_df[col] = clean_df[col].astype("string").str.strip()

clean_df["borough"] = clean_df["borough"].str.title()
clean_df["status"] = clean_df["status"].str.title()

print("Unique borough values after text cleaning:")
print(clean_df["borough"].dropna().unique())

print("\nUnique status values after text cleaning:")
print(clean_df["status"].dropna().unique())

clean_df


## Step 4: Parse Dates And Numbers Carefully

Real datasets often contain dates or numbers that do not parse cleanly.

For teaching purposes, we use `errors="coerce"` so invalid values become missing values instead of crashing the notebook. That is not a way to hide errors. It is a way to make errors measurable.

After parsing, we count how many values failed.


In [ ]:
clean_df["date_parsed"] = pd.to_datetime(clean_df["date"], errors="coerce")
clean_df["amount_numeric"] = pd.to_numeric(clean_df["amount"], errors="coerce")

failed_dates = clean_df["date_parsed"].isna().sum()
failed_amounts = clean_df["amount_numeric"].isna().sum()

print("Failed date parses:", failed_dates)
print("Failed amount parses:", failed_amounts)

clean_df[["request_id", "date", "date_parsed", "amount", "amount_numeric"]]


## Step 5: Create A Clean Output Table

The cleaned output should use a clear schema. For this tiny example:

- Keep the original `request_id`.
- Use parsed date and numeric amount fields.
- Keep standardized categorical fields.
- Preserve missing values rather than inventing replacements.

The design choice matters: changing missing values to fake categories like `Unknown` can be useful sometimes, but it must be documented.


In [ ]:
clean_output = clean_df[[
    "request_id",
    "date_parsed",
    "borough",
    "complaint_type",
    "status",
    "amount_numeric",
]].rename(columns={
    "date_parsed": "date",
    "amount_numeric": "amount",
})

clean_output


## Step 6: Verification Checks

Verification should be concrete. For Week 2, use at least these checks:

- Row count did not accidentally change.
- Expected columns exist.
- Missing values are counted.
- Parsed date and amount problems are visible.
- A few sample rows are inspected.

Assertions are useful because they fail loudly when assumptions are wrong.


In [ ]:
expected_columns = ["request_id", "date", "borough", "complaint_type", "status", "amount"]

print("Raw rows:", len(raw_df))
print("Clean rows:", len(clean_output))
print("Columns:", list(clean_output.columns))
print("Missing values after cleaning:")
print(clean_output.isna().sum())

assert len(clean_output) == len(raw_df), "Cleaning should not drop rows in this demo."
assert list(clean_output.columns) == expected_columns, "Unexpected output columns."
assert clean_output["request_id"].is_unique, "request_id should be unique in this demo."

print("Verification passed: cleaned output has expected rows, columns, and unique IDs.")
clean_output.head()


## Step 7: Save And Reload The Cleaned CSV

The assignment asks for a cleaned CSV or saved output evidence. Saving and reloading is an easy way to show that the result can move from memory into a file and back again.


In [ ]:
output_path = Path("week2_cleaned_requests.csv")
clean_output.to_csv(output_path, index=False)

reloaded_clean = pd.read_csv(output_path)

print("Saved file:", output_path)
print("Reloaded shape:", reloaded_clean.shape)
print("Reloaded columns:", list(reloaded_clean.columns))

assert len(reloaded_clean) == len(clean_output), "Reloaded row count does not match cleaned output."
assert list(reloaded_clean.columns) == list(clean_output.columns), "Reloaded columns do not match cleaned output."

reloaded_clean


## Step 8: Summarize A Small Result

A simple summary helps students see why cleaning matters.

This summary is not a full analysis. It is a demonstration that the cleaned data can support a modest question.


In [ ]:
complaint_summary = (
    clean_output
    .groupby(["borough", "complaint_type"], dropna=False)
    .size()
    .reset_index(name="request_count")
    .sort_values("request_count", ascending=False)
)

complaint_summary


In [ ]:
status_summary = (
    clean_output
    .groupby("status", dropna=False)
    .agg(
        request_count=("request_id", "count"),
        average_amount=("amount", "mean")
    )
    .reset_index()
)

status_summary


## Prompt Card: Ask For Code, But Require Verification

Students may adapt this prompt for the Week 2 assignment:

> I am working in Google Colab free tier. I have messy CSV-like text with fields [list fields]. Write beginner-friendly Python/pandas code to read the text, inspect raw rows, clean whitespace and category capitalization, parse dates, convert numeric fields, save a cleaned CSV, reload it, and print verification checks. Constraints: no shell commands, no private credentials, no large downloads, runtime under 10 minutes. Include row count, column checks, missing-value counts, sample rows, and at least one assertion.

Before using any generated answer, check:

- Did it change or drop rows without explanation?
- Did it overwrite raw values without preserving evidence?
- Did it invent missing data?
- Did it include verification?
- Can you explain every cleaning rule?


## Student Revision Activity

In the next cell, intentionally choose one revision. Examples:

- Add a rule to standardize complaint types.
- Add a check for allowed status values.
- Add a warning if too many dates fail to parse.
- Add a small chart or table for missing values.
- Add a note that explains why missing values were not filled.

The goal is to show student judgment, not just generated code.


In [ ]:
# Example student revision: define allowed status values and flag unexpected ones.

allowed_statuses = {"Open", "Closed", "Pending"}
status_values = set(clean_output["status"].dropna().unique())
unexpected_statuses = sorted(status_values - allowed_statuses)

print("Observed statuses:", sorted(status_values))
print("Unexpected statuses:", unexpected_statuses)

assert not unexpected_statuses, "Unexpected status values found. Review cleaning rules or source data."
print("Status check passed.")


## Limitation And Responsible-Use Note

Every cleaned output needs a limitation note. For this tiny sample:

- The data is invented for class demonstration.
- It has only seven rows.
- It cannot support claims about actual public service patterns.
- It is useful for practicing cleaning, verification, and documentation.

For real datasets, limitation notes should address source, sample size, missingness, definitions, privacy, and possible bias.


In [ ]:
limitation_note = pd.DataFrame([
    {
        "claim": "The notebook demonstrates a reproducible cleaning workflow.",
        "supported": True,
        "reason": "The workflow reads, cleans, saves, reloads, and verifies a small dataset."
    },
    {
        "claim": "Noise complaints are the most common real public-service issue.",
        "supported": False,
        "reason": "The sample is tiny and invented for teaching; it is not representative."
    },
    {
        "claim": "Missing and invalid values should be measured before interpretation.",
        "supported": True,
        "reason": "The notebook counts failed date and amount parsing and missing values."
    },
])

limitation_note


## Week 2 Assignment: Cleaned Mini CSV

**Instructions**

Use a messy mini text block or CSV sample and build a Colab workflow that converts it into a clean, simple CSV.

**Key points**

- Prompt framing.
- Generated-code review.
- Cleaning decisions.
- File input/output.
- Verification.

**What to do**

1. Write or revise a prompt that states the input format, desired output fields, missing-value handling, and required checks.
2. Run generated or starter code in Colab/Jupyter.
3. Revise at least one weak assumption.
4. Save the cleaned CSV.
5. Add verification cells for row count, column names, missing values, sample records, and at least one assertion.
6. Add a limitation note explaining what the cleaned output can and cannot support.

**What to submit**

- Colab notebook or `.ipynb` file.
- Prompt text.
- Cleaned CSV or saved output evidence.
- Verification output.
- Brief note naming one student revision to the AI/starter code.

**Demo readiness**

If called on, be ready to show the prompt, the cleaned output, one verification cell, and one limitation in about two minutes.


## Optional Exit Check

Run the final cell after completing the notebook. It creates a compact checklist you can submit or paste into the LMS.


In [ ]:
exit_check = pd.DataFrame([
    {"item": "Prompt included constraints and verification", "status": "check manually"},
    {"item": "Raw data loaded", "status": len(raw_df) > 0},
    {"item": "Cleaned rows match raw rows", "status": len(clean_output) == len(raw_df)},
    {"item": "Cleaned CSV saved and reloaded", "status": len(reloaded_clean) == len(clean_output)},
    {"item": "Missing values counted", "status": clean_output.isna().sum().sum() >= 0},
    {"item": "Limitation note included", "status": len(limitation_note) >= 1},
])

exit_check
